# D2 – Image Features

## 1. Import Libraries

In [1]:
import numpy as np

from library.gmm import GMM
from library.knn import KNN
from library.lda import LDA
from library.pca import PCA

## 2. Load Data

In [2]:
def load_split_data(dataset_type):
    feature_file = f"./data/image_features/{dataset_type}_data.csv"
    label_file = f"./data/image_features/{dataset_type}_label.csv"

    feature_matrix = np.loadtxt(feature_file, delimiter=",")
    target_values = np.loadtxt(label_file, delimiter=",").astype(int)

    return feature_matrix, target_values


train_features, train_labels = load_split_data("train")
valid_features, valid_labels = load_split_data("val")
test_features, test_labels = load_split_data("test")

class_labels = np.unique(train_labels)

print("Available classes:", class_labels)
print("Training samples:", train_features.shape)
print("Validation samples:", valid_features.shape)
print("Testing samples:", test_features.shape)

Available classes: [0 1 2 3 4]
Training samples: (2000, 36)
Validation samples: (500, 36)
Testing samples: (500, 36)


## 3. Evaluation

In [12]:
def build_confusion_matrix(actual, predicted, labels):
    matrix = np.zeros((len(labels), len(labels)), dtype=int)

    for row, actual_class in enumerate(labels):
        for column, predicted_class in enumerate(labels):
            matrix[row, column] = np.count_nonzero(
                (actual == actual_class) & (predicted == predicted_class)
            )

    return matrix


def evaluate_classifier(model_name, validation_actual, validation_predicted,
                        testing_actual, testing_predicted, labels):
    validation_accuracy = np.mean(validation_actual == validation_predicted)
    testing_accuracy = np.mean(testing_actual == testing_predicted)
    confusion = build_confusion_matrix(
        testing_actual, testing_predicted, labels
    )

    print(f"\\n========== {model_name} RESULTS ==========")
    print(f"Validation accuracy: {validation_accuracy:.4f}")
    print(f"Testing accuracy:    {testing_accuracy:.4f}")
    print("Test confusion matrix:")
    print(confusion)

    return {
        "Model": model_name,
        "Validation Accuracy": validation_accuracy,
        "Test Accuracy": testing_accuracy,
        "Confusion Matrix": confusion
    }


evaluation_results = []

## 4. GMM Classifier

In [4]:
gmm_classifiers = []
class_priors = []

for category in class_labels:
    category_samples = train_features[train_labels == category]

    classifier = GMM(
        n_components=3,
        regularization=1e-3,
        iterations=100,
        convergence_tol=1e-4
    )

    classifier.fit(category_samples)
    gmm_classifiers.append(classifier)

    prior_probability = len(category_samples) / len(train_features)
    class_priors.append(prior_probability)

    print(
        f"Class {category}: "
        f"EM iterations = {len(classifier.log_likelihoods)}"
    )

    if len(classifier.log_likelihoods) == classifier.iterations:
        print("Maximum iterations reached; convergence is not guaranteed.")

Class 0: EM iterations = 18
Class 1: EM iterations = 24
Class 2: EM iterations = 17
Class 3: EM iterations = 29
Class 4: EM iterations = 19


### GMM Prediction

In [5]:
log_prior_values = np.log(class_priors)

validation_log_scores = np.column_stack([
    classifier.score_samples(valid_features)
    for classifier in gmm_classifiers
])

testing_log_scores = np.column_stack([
    classifier.score_samples(test_features)
    for classifier in gmm_classifiers
])

gmm_validation_output = class_labels[
    np.argmax(validation_log_scores + log_prior_values, axis=1)
]

gmm_testing_output = class_labels[
    np.argmax(testing_log_scores + log_prior_values, axis=1)
]

### GMM Evaluation

In [6]:
gmm_results = evaluate_classifier(
    "GMM",
    valid_labels, gmm_validation_output,
    test_labels, gmm_testing_output,
    class_labels
)
evaluation_results.append(gmm_results)

\n========== GMM RESULTS ==========
Validation accuracy: 0.6800
Testing accuracy:    0.6740
Test confusion matrix:
[[79  7  5  6  3]
 [ 7 59  7 13 14]
 [ 4 18 70  4  4]
 [10 16  4 66  4]
 [ 3 19  2 13 63]]


## 5. KNN Classifier

In [7]:
knn_classifier = KNN(neighbors=5)
knn_classifier.fit(train_features, train_labels)

knn_validation_output = knn_classifier.predict(valid_features)
knn_testing_output = knn_classifier.predict(test_features)

### KNN Evaluation

In [8]:
knn_results = evaluate_classifier(
    "KNN",
    valid_labels, knn_validation_output,
    test_labels, knn_testing_output,
    class_labels
)
evaluation_results.append(knn_results)

\n========== KNN RESULTS ==========
Validation accuracy: 0.6760
Testing accuracy:    0.6840
Test confusion matrix:
[[73 15  3  7  2]
 [ 4 66  9 14  7]
 [ 1 19 75  3  2]
 [ 6 13  4 72  5]
 [ 3 26  4 11 56]]


## 6. LDA Classifier

In [9]:
lda_classifier = LDA()
lda_classifier.fit(train_features, train_labels)

lda_validation_output = lda_classifier.predict(valid_features)
lda_testing_output = lda_classifier.predict(test_features)

### LDA Evaluation

In [10]:
lda_results = evaluate_classifier(
    "LDA",
    valid_labels, lda_validation_output,
    test_labels, lda_testing_output,
    class_labels
)
evaluation_results.append(lda_results)

\n========== LDA RESULTS ==========
Validation accuracy: 0.7020
Testing accuracy:    0.6980
Test confusion matrix:
[[80  9  1  9  1]
 [ 4 71  4 10 11]
 [ 1 21 71  6  1]
 [14  9  3 60 14]
 [ 5 18  3  7 67]]


## 7. Principal Component Analysis (PCA)

In [13]:
number_of_components = 18

pca_transformer = PCA(n_components=number_of_components)
pca_transformer.fit(train_features)

train_features_pca = pca_transformer.transform(train_features)
valid_features_pca = pca_transformer.transform(valid_features)
test_features_pca = pca_transformer.transform(test_features)

print("Original feature dimensions:", train_features.shape[1])
print("PCA components:", number_of_components)
print("Training data after PCA:", train_features_pca.shape)
print("Validation data after PCA:", valid_features_pca.shape)
print("Testing data after PCA:", test_features_pca.shape)

Original feature dimensions: 36
PCA components: 18
Training data after PCA: (2000, 18)
Validation data after PCA: (500, 18)
Testing data after PCA: (500, 18)


## 8. GMM Classifier with PCA

In [14]:
gmm_pca_classifiers = []
gmm_pca_priors = []

for category in class_labels:
    category_samples = train_features_pca[train_labels == category]

    classifier = GMM(
        n_components=3,
        regularization=1e-3,
        iterations=100,
        convergence_tol=1e-4
    )

    classifier.fit(category_samples)
    gmm_pca_classifiers.append(classifier)
    gmm_pca_priors.append(len(category_samples) / len(train_features_pca))

    print(
        f"Class {category}: "
        f"EM iterations = {len(classifier.log_likelihoods)}"
    )

gmm_pca_log_priors = np.log(gmm_pca_priors)

gmm_pca_validation_scores = np.column_stack([
    classifier.score_samples(valid_features_pca)
    for classifier in gmm_pca_classifiers
])

gmm_pca_testing_scores = np.column_stack([
    classifier.score_samples(test_features_pca)
    for classifier in gmm_pca_classifiers
])

gmm_pca_validation_output = class_labels[
    np.argmax(gmm_pca_validation_scores + gmm_pca_log_priors, axis=1)
]

gmm_pca_testing_output = class_labels[
    np.argmax(gmm_pca_testing_scores + gmm_pca_log_priors, axis=1)
]

Class 0: EM iterations = 12
Class 1: EM iterations = 76
Class 2: EM iterations = 23
Class 3: EM iterations = 10
Class 4: EM iterations = 23


### GMM with PCA Evaluation

In [15]:
gmm_pca_results = evaluate_classifier(
    "GMM + PCA",
    valid_labels, gmm_pca_validation_output,
    test_labels, gmm_pca_testing_output,
    class_labels
)
evaluation_results.append(gmm_pca_results)

\n========== GMM + PCA RESULTS ==========
Validation accuracy: 0.6860
Testing accuracy:    0.7100
Test confusion matrix:
[[80  7  5  6  2]
 [ 3 68  7 16  6]
 [ 2 15 75  4  4]
 [11 10  3 72  4]
 [ 2 21  6 11 60]]


## 9. KNN Classifier with PCA

In [16]:
knn_pca_classifier = KNN(neighbors=5)
knn_pca_classifier.fit(train_features_pca, train_labels)

knn_pca_validation_output = knn_pca_classifier.predict(valid_features_pca)
knn_pca_testing_output = knn_pca_classifier.predict(test_features_pca)

### KNN with PCA Evaluation

In [17]:
knn_pca_results = evaluate_classifier(
    "KNN + PCA",
    valid_labels, knn_pca_validation_output,
    test_labels, knn_pca_testing_output,
    class_labels
)
evaluation_results.append(knn_pca_results)

\n========== KNN + PCA RESULTS ==========
Validation accuracy: 0.6960
Testing accuracy:    0.7020
Test confusion matrix:
[[73 14  1 10  2]
 [ 5 72  8 11  4]
 [ 3 14 78  3  2]
 [ 7 14  5 68  6]
 [ 1 24  5 10 60]]


## 10. LDA Classifier with PCA

In [18]:
lda_pca_classifier = LDA()
lda_pca_classifier.fit(train_features_pca, train_labels)

lda_pca_validation_output = lda_pca_classifier.predict(valid_features_pca)
lda_pca_testing_output = lda_pca_classifier.predict(test_features_pca)

### LDA with PCA Evaluation

In [19]:
lda_pca_results = evaluate_classifier(
    "LDA + PCA",
    valid_labels, lda_pca_validation_output,
    test_labels, lda_pca_testing_output,
    class_labels
)
evaluation_results.append(lda_pca_results)

\n========== LDA + PCA RESULTS ==========
Validation accuracy: 0.6960
Testing accuracy:    0.7020
Test confusion matrix:
[[78 13  1  7  1]
 [ 4 70  4 10 12]
 [ 1 22 71  5  1]
 [11 11  2 65 11]
 [ 5 15  4  9 67]]
